# 3. A day of timesteps

A scenario is one day. A **timestep** is a moment of it, and a **version** is one study state of that moment. The
two together address a snapshot, and a whole day of quarter-hourly states is a fan of timesteps hanging off the
base snapshot rather than one long line.

This notebook builds a short day, walks a single network through it running a load flow at each step, and then
crosses midnight into the next day - which is a new scenario.

The timesteps go in the way a TSO produces them: **one set of CGMES instance files per timestep**, ingested as a
difference against the state it derives from. `ssh_variant(k, label)` stands in for that export - the base archive
with its steady state file rewritten to other setpoints and another scenario time.

In [ ]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
import pypowsybl as pp

from notebook_utils import CGMES_ZIP, NEXT_DAY, SCENARIO, connect, database_url, eq_drift, fresh_scenario, next_day_zip, ssh_variant

pp.set_config_read(False)
print(pp.__version__, '->', database_url())

In [ ]:
N_TIMESTEPS = 8     # 96 in a production day; each one is a round trip to the database, so this is a sample
LABELS = ['20:00', '20:15', '20:30', '20:45', '21:00', '21:15', '21:30', '21:45'][:N_TIMESTEPS]

db = connect()
fresh_scenario(db, SCENARIO)
fresh_scenario(db, NEXT_DAY)
db.load_cgmes(CGMES_ZIP, SCENARIO, '1.0')
print(db.snapshots(SCENARIO)['timestep_label'].tolist())

## Writing the day, one file set per timestep

`load_cgmes(files, scenario, version, timestep)` on a scenario that already has a root materialises the state the
timestep derives from, compares the new files against it and stores the difference. Every timestep root hangs off
the **base** chain, which is what keeps the day a fan rather than a line: 20:15 is not reachable only through
20:00.

In [ ]:
for i, label in enumerate(LABELS, start=1):
    members = db.load_cgmes_from_binary_buffers([ssh_variant(i, label)], SCENARIO, f'1.{i}', label)
    print(label, '->', len(members), 'stored model(s)')

db.timesteps(SCENARIO)[['label', 'version_count', 'pinned_base']]

In [ ]:
db.snapshots(SCENARIO)[['version', 'timestep_label', 'kind', 'depth', 'fast']]

## Walking the day

One network, one object, one load flow per timestep.

In [ ]:
rows = []
walker = pp.network.from_rdf_db(db, SCENARIO, '1.0')

for label in LABELS:
    start = time.perf_counter()
    route = walker.update_from_rdf_db(db, SCENARIO, None, label)
    seconds = time.perf_counter() - start
    result = pp.loadflow.run_ac(walker)
    rows.append({'scenario': SCENARIO, 'label': label, 'route': route,
                 'total_load_p': float(walker.get_loads()['p0'].sum()),
                 'lf_status': str(result[0].status), 'seconds': seconds})

day = pd.DataFrame(rows)
day

In [ ]:
ax = day.set_index('label')['total_load_p'].plot(marker='o')
ax.set_ylabel('total load p0 [MW]')
ax.set_title('A day of timesteps')
plt.tight_layout()

Every step is a `diff`: the change touches only steady-state predicates, so the network is updated in place.

## The day as variants

The walk above keeps no history: after the last timestep the first one is gone. Loading every timestep as a
network of its own keeps all of them, but converts the same equipment once per timestep.

The third way is one network with one **variant per timestep**. `from_rdf_db(..., timesteps=[...])` converts the
first snapshot, clones it once per further timestep and applies the stored differences on the clones - one chain
query, one statement fetch and one conversion, whatever the number of timesteps. The variants are named after the
timestep labels.

In [ ]:
variants = pp.network.from_rdf_db(db, SCENARIO, None, timesteps=LABELS)
print(sorted(variants.get_variant_ids()))
print('working variant:', variants.get_working_variant_id())

In [ ]:
variants.variants_binding()[['version', 'timestep', 'label', 'cloned_from', 'status']]

`status` says what each variant stands for: `primary` is the network's own identity (a bulk load leaves it at the
first requested snapshot), `bound` is a variant that stands for a snapshot, `refused` is a snapshot that could not
be reached - see below.

A study now means selecting a variant and running whatever you run. No reload, no second network.

In [ ]:
per_variant = []
for label in LABELS:
    variants.set_working_variant(label)
    status = pp.loadflow.run_ac(variants)[0].status
    per_variant.append({'label': label, 'total_load_p': float(variants.get_loads()['p0'].sum()),
                        'lf_status': str(status)})
variants.set_working_variant('InitialState')

per_variant = pd.DataFrame(per_variant)
pd.testing.assert_series_equal(per_variant['total_load_p'], day['total_load_p'], check_exact=False, rtol=1e-9)
print('every variant carries the state the walk saw at that timestep')
per_variant

## What a variant cannot be

IIDM stores only part of a grid model per variant. Setpoints, switch positions, tap positions and terminal
connections are per variant; **branch impedances, operational limit values, voltage limits and the equipment model
itself are not**. A difference that writes one of those would change every other variant of the network at the
same time, so it is refused - with reasons, and with the network left exactly as it was.

The archive below is one such timestep: the same setpoint change as the others, plus a renamed line.

In [ ]:
db.load_cgmes_from_binary_buffers([eq_drift(9, '22:00')], SCENARIO, '1.9', '22:00')

try:
    variants.update_from_rdf_db(db, SCENARIO, None, '22:00', variant='22:00')
    print('not reached')
except pp.network.RdfDbVariantRefusedError as refusal:
    print('refused:', refusal.variant)
    for reason in refusal.reasons:
        print(' -', reason)

print(sorted(variants.get_variant_ids()), '<- the refused variant was not created')

In [ ]:
# The way out is a network of its own; the multi-variant network is untouched
drifted = pp.network.from_rdf_db(db, SCENARIO, None, '22:00')
print(sorted(n for n in drifted.get_lines()['name'] if n.startswith('drifted')))

**Memory.** A variant costs one slot in every per-variant array - setpoints, switch states, terminal and bus state
variables - not a copy of the topology. On the MicroGrid test configuration the core benchmark measures about
**1 MB for 96 variants**; a large model scales with the number of per-variant values, not with the file size. The
figure is a heap reading after a collection, so treat it as an order of magnitude rather than an exact number.

## Random access

A timestep is an address, not a position in a walk: reading it directly gives the same state.

In [ ]:
direct = pp.network.from_rdf_db(db, SCENARIO, None, LABELS[-1])
pd.testing.assert_frame_equal(direct.get_loads()[['p0', 'q0']].sort_index(),
                              walker.get_loads()[['p0', 'q0']].sort_index())
print('the walked state and the directly loaded state agree')

## Checkpoints

A checkpoint folds the differences of a snapshot into full graphs, so that loading it needs no walk down the chain.
It changes nothing about what the snapshot is.

In [ ]:
before = pp.network.from_rdf_db(db, SCENARIO, None, LABELS[-1]).get_loads()[['p0']].sort_index()
iri = db.checkpoint(SCENARIO, None, LABELS[-1])
print(bool(db.snapshots(SCENARIO).loc[iri, 'has_full']))
after = pp.network.from_rdf_db(db, SCENARIO, None, LABELS[-1]).get_loads()[['p0']].sort_index()
pd.testing.assert_frame_equal(before, after)
print('same network before and after the checkpoint')

## Crossing midnight

The next day is a new scenario. The walk continues, but the first step into it is a **full** reload - differences
never cross scenarios - and the steps inside it are differences again.

In [ ]:
db.load_cgmes_from_binary_buffers([next_day_zip()], NEXT_DAY, '1.0')
db.load_cgmes_from_binary_buffers([ssh_variant(1, '00:15', NEXT_DAY)], NEXT_DAY, '1.1', '00:15')

db.timesteps(NEXT_DAY)[['label', 'version_count']]

In [ ]:
for scenario, version, label in [(NEXT_DAY, '1.0', None), (NEXT_DAY, '1.1', '00:15')]:
    start = time.perf_counter()
    route = walker.update_from_rdf_db(db, scenario, version, label)
    rows.append({'scenario': scenario, 'label': label or 'base', 'route': route,
                 'total_load_p': float(walker.get_loads()['p0'].sum()),
                 'lf_status': str(pp.loadflow.run_ac(walker)[0].status),
                 'seconds': time.perf_counter() - start})

pd.DataFrame(rows)[['scenario', 'label', 'route', 'total_load_p', 'seconds']]

In [ ]:
db.scenarios()

The `route` column tells the whole story: `diff` inside a day, `full` at the day boundary. Note also that the same
label means two different moments in the two scenarios - `db.timesteps(...)` shows the resolved instants.

In [ ]:
db.close()